# Análise financeira

Análises exploratórias sobre a tabela `transactions` (conta corrente do Inter).

Antes de rodar:
- Ative o ambiente virtual e aplique as migrations (veja o README).
- O MySQL precisa estar rodando com os dados importados.

**Convenções usadas em todo o notebook:**
- **Investimento:** aplicações, resgates e Tesouro Direto (inclusive `Debito Online Td`) são movimentação de patrimônio,
  não receita nem despesa. Ficam separados (seção 9).
- **Rotina:** transações sem `context`. Viagens e projetos marcados com `set-context`
  ficam de fora das análises mensais e aparecem na seção 10.
- **Valor:** é usado `value`, que já considera os ajustes (`adjust`), e não `original_value`.
- **Últimos 12 meses:** contados até o último mês completo com dados.

> As saídas deste notebook contêm seus dados financeiros. Use o `nbstripout` (veja o README)
> para não commitá-las.

In [ ]:
import re

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sqlalchemy import text

from financial.database import get_engine

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_colwidth = 60

## Estilo dos gráficos

Paleta categórica em ordem fixa, validada para daltonismo. Cada entidade mantém a mesma cor
em todos os gráficos (por exemplo, receita é sempre azul e despesa sempre laranja).

In [ ]:
AZUL, LARANJA, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
VERMELHO = "#e34948"
CINZA = "#b5b3ad"
TEXTO, TEXTO_SECUNDARIO = "#0b0b0b", "#52514e"
SUPERFICIE, GRADE, EIXO = "#fcfcfb", "#ecebe7", "#d9d8d3"
AZUL_SEQUENCIAL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

pio.templates["financas"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, system-ui, sans-serif", size=13, color=TEXTO_SECUNDARIO),
    title=dict(font=dict(size=16, color=TEXTO), x=0, xanchor="left"),
    paper_bgcolor=SUPERFICIE,
    plot_bgcolor=SUPERFICIE,
    colorway=[AZUL, LARANJA, AQUA, "#eda100", "#e87ba4", "#008300", "#4a3aa7", VERMELHO],
    separators=",.",
    xaxis=dict(showgrid=False, linecolor=EIXO, ticks="", title=None),
    yaxis=dict(gridcolor=GRADE, gridwidth=1, zeroline=True, zerolinecolor=EIXO,
               tickformat=",.0f", title=None),
    legend=dict(orientation="h", x=0, y=1.02, yanchor="bottom", title=None),
    hovermode="x unified",
    bargap=0.35,
    barcornerradius=4,
    margin=dict(l=8, r=8, t=64, b=8),
))
pio.templates.default = "plotly_white+financas"


def brl(valor: float) -> str:
    texto = f"{abs(valor):,.2f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return f"-R$ {texto}" if valor < 0 else f"R$ {texto}"


def linhas(fig):
    fig.update_traces(line=dict(width=2), selector=dict(type="scatter"))
    return fig

## Carregando os dados

In [ ]:
engine = get_engine()

df = pd.read_sql(text("""
    select
        t.id, t.date, t.description, t.value, t.original_value, t.balance,
        t.context, c.name as categoria, c.sector as setor
    from transactions t
    left join categories c on c.id = t.category_id
    order by t.date, t.id
"""), engine, parse_dates=["date"])

df[["value", "original_value", "balance"]] = df[["value", "original_value", "balance"]].astype(float)

PADRAO_INVESTIMENTO = r"^(?:aplica[cç][aã]o|resgate)\b|tesouro direto|^debito online td\b"
investimento = df["description"].str.contains(PADRAO_INVESTIMENTO, case=False, regex=True)
df["tipo"] = "Despesa"
df.loc[df["value"] > 0, "tipo"] = "Receita"
df.loc[investimento, "tipo"] = "Investimento"

df["mes"] = df["date"].dt.to_period("M").dt.to_timestamp()
df["ano"] = df["date"].dt.year
df["categoria"] = df["categoria"].fillna("Sem categoria")
df["setor"] = df["setor"].fillna("Sem categoria")


def estabelecimento(descricao: str) -> str:
    """Extrai o nome do estabelecimento ou da pessoa da descrição do Inter."""
    d = descricao
    d = re.sub(r"^[^:]{0,30}:\s*", "", d) if ":" in d[:30] else re.sub(r"^[^-]{0,30}-\s*", "", d)
    d = re.sub(r"(?i)(compra\s+)?no estabelecimento\s*", "", d)
    d = re.sub(r"(?i)^\"?cp\s*:\s*\d+-", "", d.strip('" '))
    d = re.sub(r"^[\d\s]+(?=[A-Za-z])", "", d)
    return re.sub(r"\s+", " ", d).strip('" ')[:40].strip()


df["estabelecimento"] = df["description"].map(estabelecimento)

rotina = df[df["context"].isna()]
despesas = rotina[rotina["tipo"] == "Despesa"].assign(gasto=lambda d: -d["value"])

ultima_data = df["date"].max()
ultimo_mes_completo = (ultima_data + pd.Timedelta(days=1)).to_period("M").to_timestamp() - pd.DateOffset(months=1)
inicio_12m = ultimo_mes_completo - pd.DateOffset(months=11)
ultimos_12m = lambda d: d[(d["mes"] >= inicio_12m) & (d["mes"] <= ultimo_mes_completo)]

SETORES = sorted(s for s in df["setor"].unique() if s != "Sem categoria") + ["Sem categoria"]
CORES_SETOR = dict(zip(SETORES, [AZUL, LARANJA, AQUA][:len(SETORES) - 1] + [CINZA]))

print(f"{len(df):,} transações de {df['date'].min():%d/%m/%Y} a {ultima_data:%d/%m/%Y}")
print(f"Últimos 12 meses completos: {inicio_12m:%m/%Y} a {ultimo_mes_completo:%m/%Y}")
df["tipo"].value_counts()

## 1. Cobertura dos dados

Antes de qualquer conclusão: quais meses têm dados? Meses vazios distorcem médias e totais.

In [ ]:
todos_meses = pd.date_range(df["mes"].min(), df["mes"].max(), freq="MS")
por_mes = df.groupby("mes").size().reindex(todos_meses, fill_value=0)

faltando = por_mes[por_mes == 0]
print("Meses sem nenhuma transação:", ", ".join(f"{m:%m/%Y}" for m in faltando.index) or "nenhum")

grade = por_mes.to_frame("qtd").assign(ano=lambda d: d.index.year, mes=lambda d: d.index.month)
grade = grade.pivot(index="ano", columns="mes", values="qtd").replace(0, float("nan"))

fig = px.imshow(
    grade, color_continuous_scale=AZUL_SEQUENCIAL, aspect="auto",
    labels=dict(x="Mês", y="Ano", color="Transações"),
    title="Transações por mês (células vazias = sem dados)",
)
fig.update_xaxes(tickmode="array", tickvals=list(range(1, 13)), side="top")
fig.update_yaxes(tickmode="array", tickvals=list(grade.index), gridcolor=SUPERFICIE)
fig.update_layout(hovermode="closest", plot_bgcolor=SUPERFICIE)
fig.show()

## 2. Saldo da conta ao longo do tempo

Saldo da conta corrente no fim de cada dia. Dinheiro aplicado em CDB ou Tesouro não aparece aqui.

In [ ]:
saldo = df.groupby("date")["balance"].last()

fig = linhas(px.line(saldo, title="Saldo da conta corrente", render_mode="svg"))
fig.update_traces(hovertemplate="%{x|%d/%m/%Y}<br>R$ %{y:,.2f}<extra></extra>")
fig.update_layout(showlegend=False, hovermode="closest")
fig.show()

print(f"Saldo atual: {brl(saldo.iloc[-1])} | máximo: {brl(saldo.max())} em {saldo.idxmax():%d/%m/%Y}")

## 3. Receitas x despesas

Rotina sem investimentos. "Receita" inclui tudo que entrou na conta: salário,
Pix recebidos (inclusive reembolsos de amigos) e rendimentos de FIIs.

A fatura do cartão de crédito entra como uma única despesa no dia do pagamento. O detalhe das
compras no cartão não está no extrato da conta corrente.

In [ ]:
fluxo = (
    rotina[rotina["tipo"] != "Investimento"]
    .pivot_table(index="mes", columns="tipo", values="value", aggfunc="sum")
    .reindex(todos_meses).fillna(0)
)
fluxo["Despesa"] = -fluxo["Despesa"]
fluxo["Resultado"] = fluxo["Receita"] - fluxo["Despesa"]

fig = linhas(px.line(
    fluxo, y=["Receita", "Despesa"], title="Receitas e despesas por mês",
    color_discrete_map={"Receita": AZUL, "Despesa": LARANJA},
))
fig.update_traces(hovertemplate="R$ %{y:,.2f}")
fig.show()

fig = px.bar(
    fluxo, y="Resultado", title="Resultado do mês (receitas − despesas)",
    color=fluxo["Resultado"] >= 0, color_discrete_map={True: AZUL, False: VERMELHO},
)
fig.update_traces(hovertemplate="%{x|%m/%Y}<br>R$ %{y:,.2f}<extra></extra>")
fig.update_layout(showlegend=False, hovermode="closest")
fig.show()

In [ ]:
anual = fluxo.groupby(fluxo.index.year)[["Receita", "Despesa", "Resultado"]].sum()
anual["Meses com dados"] = por_mes.groupby(por_mes.index.year).apply(lambda s: (s > 0).sum())
anual["Taxa de poupança"] = (anual["Resultado"] / anual["Receita"]).map("{:.0%}".format)
anual["Despesa média/mês"] = anual["Despesa"] / anual["Meses com dados"]
anual.index.name = "Ano"
anual

## 4. Despesas por setor

Setores das categorias (essencial, menos essencial, luxo). "Sem categoria" mostra o quanto
das despesas ainda não tem regra de categoria (veja a seção 7).

In [ ]:
por_setor = despesas.pivot_table(index="ano", columns="setor", values="gasto", aggfunc="sum").fillna(0)
por_setor = por_setor.reindex(columns=[s for s in SETORES if s in por_setor.columns])

fig = px.bar(
    por_setor, title="Despesas por setor e ano", color_discrete_map=CORES_SETOR,
)
fig.update_traces(marker_line_width=2, marker_line_color=SUPERFICIE, hovertemplate="R$ %{y:,.2f}")
fig.update_xaxes(type="category")
fig.show()

(por_setor.div(por_setor.sum(axis=1), axis=0) * 100).round(1).add_suffix(" (%)")

## 5. Maiores categorias nos últimos 12 meses

In [ ]:
categorizadas = ultimos_12m(despesas)
sem_categoria = categorizadas.loc[categorizadas["categoria"] == "Sem categoria", "gasto"].sum()
top = (
    categorizadas[categorizadas["categoria"] != "Sem categoria"]
    .groupby("categoria")["gasto"].sum().nlargest(15).sort_values()
)

fig = px.bar(
    top, orientation="h", text=top.map(brl),
    title=f"Despesas por categoria ({inicio_12m:%m/%Y} a {ultimo_mes_completo:%m/%Y})"
          f" · fora do gráfico: {brl(sem_categoria)} sem categoria",
)
fig.update_traces(marker_color=AZUL, textposition="outside", cliponaxis=False,
                  hovertemplate="%{y}<br>R$ %{x:,.2f}<extra></extra>")
fig.update_layout(showlegend=False, hovermode="closest", height=520)
fig.update_xaxes(showticklabels=False, title=None)
fig.update_yaxes(title=None, gridcolor=SUPERFICIE)
fig.show()

## 6. Evolução das 4 maiores categorias

Um gráfico por categoria (mesma escala), para comparar tendências sem misturar cores.

In [ ]:
quatro = top.nlargest(4).index.tolist()
evolucao = (
    despesas[despesas["categoria"].isin(quatro)]
    .pivot_table(index="mes", columns="categoria", values="gasto", aggfunc="sum")
    .reindex(todos_meses).fillna(0)[quatro]
    .reset_index(names="mes").melt(id_vars="mes", var_name="categoria", value_name="gasto")
)

fig = linhas(px.line(
    evolucao, x="mes", y="gasto", facet_col="categoria", facet_col_wrap=2,
    title="Gasto mensal por categoria", height=520,
))
fig.update_traces(line_color=AZUL, hovertemplate="%{x|%m/%Y}<br>R$ %{y:,.2f}<extra></extra>")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font_color=TEXTO))
fig.update_yaxes(title=None)
fig.update_xaxes(title=None)
fig.update_layout(hovermode="closest", showlegend=False)
fig.show()

## 7. Qualidade da categorização

Quanto das despesas está sem categoria, e quais estabelecimentos sem categoria mais pesam.
Esses são os melhores candidatos para novas regras (`create-category-rule`).

In [ ]:
cobertura = despesas.assign(sem=despesas["categoria"] == "Sem categoria").groupby("ano").apply(
    lambda d: pd.Series({
        "Despesas": d["gasto"].sum(),
        "Sem categoria": d.loc[d["sem"], "gasto"].sum(),
    }), include_groups=False,
)
cobertura["% sem categoria"] = (cobertura["Sem categoria"] / cobertura["Despesas"]).map("{:.0%}".format)
cobertura

In [ ]:
(
    ultimos_12m(despesas)
    .query("categoria == 'Sem categoria'")
    .groupby("estabelecimento")
    .agg(qtd=("gasto", "size"), total=("gasto", "sum"), ultima=("date", "max"))
    .sort_values("total", ascending=False)
    .head(25)
)

## 8. Gastos recorrentes

Estabelecimentos com despesa em pelo menos 6 dos últimos 12 meses: contas fixas, assinaturas
e hábitos. O coeficiente de variação baixo indica valor estável (típico de assinatura).

In [ ]:
recorrentes = (
    ultimos_12m(despesas)
    .groupby("estabelecimento")
    .agg(
        meses=("mes", "nunique"),
        vezes=("gasto", "size"),
        total=("gasto", "sum"),
        valor_medio=("gasto", "mean"),
        variacao=("gasto", lambda g: g.std(ddof=0) / g.mean() if g.mean() else 0),
        categoria=("categoria", "first"),
    )
    .query("meses >= 6")
    .assign(media_mensal=lambda d: d["total"] / 12)
    .sort_values("total", ascending=False)
)
recorrentes["variacao"] = recorrentes["variacao"].map("{:.0%}".format)
recorrentes

## 9. Investimentos

In [ ]:
inv = rotina[rotina["tipo"] == "Investimento"].copy()
inv["movimento"] = inv["value"].gt(0).map({True: "Resgate", False: "Aplicação"})

inv_anual = inv.pivot_table(index="ano", columns="movimento", values="value", aggfunc="sum").fillna(0)
inv_anual["Aplicação"] = -inv_anual.get("Aplicação", 0)
inv_anual["Aplicado líquido"] = inv_anual["Aplicação"] - inv_anual.get("Resgate", 0)

fig = px.bar(
    inv_anual, y=["Aplicação", "Resgate"], barmode="group",
    title="Aplicações e resgates por ano", color_discrete_map={"Aplicação": AZUL, "Resgate": LARANJA},
)
fig.update_traces(hovertemplate="R$ %{y:,.2f}")
fig.update_xaxes(type="category")
fig.show()

inv_anual.index.name = "Ano"
inv_anual

## 10. Viagens e projetos (contextos)

Transações marcadas com `set-context`, fora das análises de rotina.

In [ ]:
(
    df[df["context"].notna()]
    .groupby("context")
    .agg(qtd=("value", "size"), total=("value", "sum"), inicio=("date", "min"), fim=("date", "max"))
    .sort_values("total")
)

## 11. Maiores despesas dos últimos 12 meses

In [ ]:
(
    ultimos_12m(despesas)
    .nlargest(15, "gasto")[["date", "description", "categoria", "gasto"]]
    .reset_index(drop=True)
)

## 12. Sazonalidade

Gasto médio por mês do ano, considerando só os meses que têm dados.
Mostra em que época do ano as despesas costumam subir.

In [ ]:
meses_com_dados = por_mes[por_mes > 0].index
gasto_mensal = despesas.groupby("mes")["gasto"].sum().reindex(meses_com_dados).fillna(0)
sazonal = gasto_mensal.groupby(gasto_mensal.index.month).mean()
NOMES_MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
sazonal.index = [NOMES_MESES[m - 1] for m in sazonal.index]

fig = px.bar(sazonal, title="Gasto médio por mês do ano")
fig.update_traces(marker_color=AZUL, hovertemplate="%{x}<br>R$ %{y:,.2f}<extra></extra>")
fig.update_layout(showlegend=False, hovermode="closest")
fig.show()